# Introduction

This Notebook demonstrates how to build a hybrid recommender system (combining a item-based collaborative filtering and a content-based recommender system).  

We will use MovieLens dataset.

# Data preparation

In [1]:
import os
import torch
import pandas as pd
import torch.nn.functional as F

In [2]:
def load_movies(path):
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            movie_id, title, genres = line.strip().split("::")

            rows.append({
                "movie_id": int(movie_id),
                "title": title,
                "genres": genres
            })

    return pd.DataFrame(rows)


def load_ratings(path):
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, movie_id, rating, timestamp = line.strip().split("::")

            rows.append({
                "user_id": int(user_id),
                "movie_id": int(movie_id),
                "rating": float(rating)
            })

    return pd.DataFrame(rows)

In [3]:
root_path = "/kaggle/input/datasets/sherinclaudia/movielens"
ratings_path = os.path.join(root_path, "ratings.dat")
movies_path = os.path.join(root_path, "movies.dat")

In [4]:
ratings_df = load_ratings(ratings_path)
movies_df = load_movies(movies_path)

 # Hybrid recommender system

First, we build the user–item matrix used for item-based collaborative filtering.

In [5]:
user_item_df = ratings_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating",
    fill_value=0
)

ratings = torch.tensor(
    user_item_df.values,
    dtype=torch.float32
)

Next, we compute item–item similarity from user ratings.

In [6]:
item_vectors = ratings.T

normalized_items = F.normalize(item_vectors, p=2, dim=1)

item_similarity = normalized_items @ normalized_items.T

Now we build the content-based representation using movie genres.

In [7]:
all_genres = sorted(
    set(
        genre
        for genres in movies_df["genres"]
        for genre in genres.split("|")
        if genre != "(no genres listed)"
    )
)

genre_to_idx = {genre: idx for idx, genre in enumerate(all_genres)}

movie_features = torch.zeros(
    (len(movies_df), len(all_genres)),
    dtype=torch.float32
)

for row_idx, genres in enumerate(movies_df["genres"]):
    for genre in genres.split("|"):
        if genre in genre_to_idx:
            movie_features[row_idx, genre_to_idx[genre]] = 1.0

normalized_movie_features = F.normalize(movie_features, p=2, dim=1)

We need mappings between MovieLens movie IDs and matrix indices.

In [8]:
movie_id_to_content_index = {
    movie_id: idx
    for idx, movie_id in enumerate(movies_df["movie_id"])
}

movie_id_to_cf_index = {
    movie_id: idx
    for idx, movie_id in enumerate(user_item_df.columns)
}

cf_index_to_movie_id = {
    idx: movie_id
    for movie_id, idx in movie_id_to_cf_index.items()
}

Finally, we define the hybrid recommender.

Note: for normalization of collaborative filtering and content scores, we will use `min_max_normalize` function implemented below (the normalize function from PyTorch will give small scores that might be misleading):

In [9]:
def min_max_normalize(scores):
    min_score = scores.min()
    max_score = scores.max()

    return (scores - min_score) / torch.clamp(
        max_score - min_score,
        min=1e-8
    )

In [10]:
def recommend_hybrid(
    user_id,
    ratings_df,
    movies_df,
    ratings,
    user_item_df,
    item_similarity,
    normalized_movie_features,
    movie_id_to_cf_index,
    movie_id_to_content_index,
    cf_index_to_movie_id,
    alpha=0.7,
    top_k=10
):
    user_idx = list(user_item_df.index).index(user_id)

    user_ratings = ratings[user_idx]

    # Item-based collaborative filtering score
    cf_scores = user_ratings @ item_similarity

    rated_mask = user_ratings > 0
    normalization = rated_mask.float() @ item_similarity

    cf_scores = cf_scores / torch.clamp(normalization, min=1e-8)

    # Content-based score
    user_history = ratings_df[ratings_df["user_id"] == user_id]

    liked_movie_ids = user_history[
        user_history["rating"] >= 4.0
    ]["movie_id"].tolist()

    liked_content_indices = [
        movie_id_to_content_index[movie_id]
        for movie_id in liked_movie_ids
        if movie_id in movie_id_to_content_index
    ]

    user_profile = normalized_movie_features[liked_content_indices].mean(dim=0)

    content_scores_all = normalized_movie_features @ user_profile

    # Align content scores to collaborative filtering movie order
    content_scores = torch.zeros_like(cf_scores)

    for cf_idx in range(len(cf_scores)):
        movie_id = cf_index_to_movie_id[cf_idx]

        if movie_id in movie_id_to_content_index:
            content_idx = movie_id_to_content_index[movie_id]
            content_scores[cf_idx] = content_scores_all[content_idx]

    # Normalize both scores before combining
    cf_scores = min_max_normalize(cf_scores)
    content_scores = min_max_normalize(content_scores)

    
    # cf_scores = F.normalize(cf_scores.unsqueeze(0), p=2, dim=1).squeeze()
    # content_scores = F.normalize(content_scores.unsqueeze(0), p=2, dim=1).squeeze()

    hybrid_scores = alpha * cf_scores + (1 - alpha) * content_scores

    # Do not recommend movies already rated
    hybrid_scores[rated_mask] = -1

    top_indices = torch.topk(hybrid_scores, top_k).indices.tolist()

    recommended_movie_ids = [
        cf_index_to_movie_id[idx]
        for idx in top_indices
    ]

    recommendations = movies_df[
        movies_df["movie_id"].isin(recommended_movie_ids)
    ][["movie_id", "title", "genres"]].copy()

    score_map = {
        cf_index_to_movie_id[idx]: hybrid_scores[idx].item()
        for idx in top_indices
    }

    recommendations["hybrid_score"] = recommendations["movie_id"].map(score_map)

    recommendations = recommendations.sort_values(
        "hybrid_score",
        ascending=False
    )

    return recommendations

We can now request recommendations for a specific user.

In [11]:
recommendations = recommend_hybrid(
    user_id=1,
    ratings_df=ratings_df,
    movies_df=movies_df,
    ratings=ratings,
    user_item_df=user_item_df,
    item_similarity=item_similarity,
    normalized_movie_features=normalized_movie_features,
    movie_id_to_cf_index=movie_id_to_cf_index,
    movie_id_to_content_index=movie_id_to_content_index,
    cf_index_to_movie_id=cf_index_to_movie_id,
    alpha=0.7,
    top_k=10
)

print(recommendations)

      movie_id                                        title  \
3140      3209                  Loves of Carmen, The (1948)   
1846      1915  Voyage to the Beginning of the World (1997)   
1001      1014                             Pollyanna (1960)   
977        989    Schlafes Bruder (Brother of Sleep) (1995)   
33          34                                  Babe (1995)   
745        755                                   Kim (1950)   
1748      1812                            Wide Awake (1998)   
2776      2845                            White Boys (1999)   
2010      2079                             Kidnapped (1960)   
2274      2343                        Naked Man, The (1998)   

                       genres  hybrid_score  
3140                    Drama      0.957685  
1846                    Drama      0.957685  
1001  Children's|Comedy|Drama      0.892016  
977                     Drama      0.891666  
33    Children's|Comedy|Drama      0.891046  
745          Children's|Drama 